# Direk Kaggle Worker — Qwen 2.5 3B (T3 heavy batch)

Ang notebook na ito ang **T3 heavy-batch worker** ng MOVIE CREATOR. Dito tumatakbo
ang Qwen 2.5 3B sa GPU ng Kaggle, at ito ang sinusubukan ng app bago bumagsak sa
lokal na Ollama (T1).

**Ang kontrata ng app** (`src/services/pilotService.ts`) — ito lang ang tinatawag:

| Route | Body | Sagot |
|---|---|---|
| `GET  /health` | — | `{ ok, model, model_ready }` |
| `GET  /v1/models` | — | listahan ng modelo |
| `POST /generate_script` | `{ brief }` | `{ script }` |
| `POST /generate_character_bible` | `{ script }` | `{ bible }` |
| `POST /breakdown_scenes` | `{ script }` | `{ scenes: [{ heading, narration, visual, mood }] }` |
| `POST /v1/chat/completions` | OpenAI shape | OpenAI shape |

**Bawal ang mag-imbento.** Kapag wala ang modelo o nabigo ang pagbuo, sasagot ito
ng **503** na may `code: "AI_UNAVAILABLE"`. Hindi kailanman gagawa ng pekeng
screenplay ang worker — ginagawa iyon ng app na `PILOT_UNAVAILABLE`.

## Bakit may "tagutol ng Tagalog" (cell 4)

Sumulat ang 3B modelo ng ganito sa totoong takbo:

> "Nakakakaban talaga ang mga sariling puso sa mga bayad na lumipas."
> "Masamang kasintahan ang mga pananampalataya na walang magbabago."
> "pagkakaiba-biba"

Hindi ito kaso ng maling prompt lang. May dalawang tunay na dahilan:

1. **Nakakalason sa Tagalog ang `repetition_penalty=1.1`.** Pinaparusa nito ang
   mga token na paulit-ulit. Sa Tagalog, ang pinaka-madalas na token ay ang mga
   pantulong na salita — `ang`, `ng`, `sa`, `na`, `ay`, `si`. Kaya pinipilit ng
   parusa ang modelo na ipalit ang mga ito ng bihira at maling salita. Iyon ang
   eksaktong itsura ng basura sa itaas. Naka-`1.0` na ito ngayon, ibig sabihin
   patay.
2. **Malayang sanaysay ang hiningi, hindi screenplay.** Ang hinihingi ng app ay
   eksena na may `heading`/`narration`/`visual`/`mood`, hindi "maikling kwento".


In [ ]:
# 1. I-install muna ang mga pakete bago mag-import.
import os
import subprocess

subprocess.run(
    ["pip", "install", "-q", "fastapi", "uvicorn", "transformers", "accelerate", "pydantic", "requests", "pandas"],
    check=True,
)
print("✅ Naka-install na ang mga pakete.")

# 2. Kunin ang mga sikreto sa Kaggle Secrets. Walang naka-hardcode na key.
try:
    from kaggle_secrets import UserSecretsClient

    SECRETS = UserSecretsClient()
    SECRET_NAMES = [
        "HF_TOKEN",
        "FISHAUDIO_API_KEY",
        "CF_TUNNEL_TOKEN",
        "CF_TUNNEL_HOSTNAME",
        "GEMINI_API_KEY",
        "KAGGLE_PILOT_URL",
        "PILOT_SYNC_TOKEN",
        "APP_URL",
    ]
    for name in SECRET_NAMES:
        try:
            value = SECRETS.get_secret(name)
        except Exception:
            print(f"⚠️ Wala ang sikreto, nilaktawan: {name}")
            continue
        if value:
            os.environ[name] = value
            print(f"✅ Nakuha ang sikreto: {name}")
except ImportError:
    print("⚠️ Hindi ito Kaggle; nilaktawan ang UserSecretsClient.")


In [ ]:
# 3. Sangguniang katalogo at ang gate ng kulang na detalye.
import os
import re

import pandas as pd

NETFLIX_PATH = "/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv"
COLUMNS = ["title", "type", "listed_in", "release_year", "duration"]

if os.path.exists(NETFLIX_PATH):
    CATALOG = pd.read_csv(NETFLIX_PATH, usecols=COLUMNS)
    print(f"✅ Nakarga ang sangguniang katalogo: {len(CATALOG)} hilera.")
else:
    CATALOG = pd.DataFrame(columns=COLUMNS)
    print("⚠️ Hindi nakita ang Netflix CSV; tatakbo pa rin nang walang sanggunian.")

# Hinahanap ang uri bilang BUONG salita. Kung substring lang, maituturing na
# "action" ang "satisfaction" at magiging maling uri ang sagot.
GENRE_MAP = {
    "aksyon": "action", "action": "action",
    "drama": "drama",
    "komedya": "comedies", "comedy": "comedies",
    "katatakutan": "horror", "horror": "horror",
    "thriller": "thriller",
    "science fiction": "sci-fi", "sci-fi": "sci-fi", "sci fi": "sci-fi",
    "romantiko": "romantic", "romance": "romantic", "romantic": "romantic",
    "love story": "romantic",
}

MOOD_MAP = {
    "masaya": "masaya",
    "malungkot": "malungkot",
    "nakakakaba": "nakakakaba",
    "mabigat": "mabigat",
    "magaan": "magaan",
    "madilim": "madilim",
    "romantiko": "romantiko",
    "makabagbag-damdamin": "makabagbag-damdamin",
    "nakakatawa": "nakakatawa",
    "nakapupukaw": "nakapupukaw",
    "nostalhiko": "nostalhiko",
}

DURATION_RE = re.compile(r"\d+\s*(segundo|minuto|oras|sec|min|hour)", re.I)

QUESTIONS = {
    "uri": "Anong uri ng pelikula ang gusto mo? Halimbawa: aksyon, drama, komedya, katatakutan, o romantiko.",
    "haba": "Gaano katagal ang gusto mong pelikula? Halimbawa: 30 segundo, 3 minuto, o 45 minuto.",
    "mood": "Anong damdamin ang gusto mong maramdaman ng manonood? Halimbawa: masaya, malungkot, nakakakaba, o mabigat.",
}


def whole_word(word, text):
    return re.search(rf"(?<!\w){re.escape(word)}(?!\w)", text) is not None


def find_genre(text):
    lowered = (text or "").lower()
    for word, genre in GENRE_MAP.items():
        if whole_word(word, lowered):
            return genre
    return None


def find_mood(text):
    lowered = (text or "").lower()
    for word, mood in MOOD_MAP.items():
        if whole_word(word, lowered):
            return mood
    return None


def find_duration(text):
    match = DURATION_RE.search((text or "").lower())
    return match.group(0) if match else None


def missing_fields(text):
    """Ang kulang sa utos. Kasama ang uri: walang uri, walang pelikula.

    Kung hindi hihingin ang uri, mag-iimbento ang modelo ng sariling uri at
    doon nagsisimula ang basura.
    """
    missing = []
    if not find_genre(text):
        missing.append("uri")
    if not find_duration(text):
        missing.append("haba")
    if not find_mood(text):
        missing.append("mood")
    return missing


def search_titles(genre, kind=None, limit=5):
    """Totoong pamagat mula sa katalogo, para may sanggunian ang modelo."""
    if CATALOG.empty or not genre:
        return []
    rows = CATALOG
    if kind:
        rows = rows[rows["type"].str.lower() == kind.lower()]
    rows = rows[rows["listed_in"].fillna("").str.lower().str.contains(genre.lower(), regex=False)]
    return rows.head(max(1, min(int(limit), 5))).to_dict(orient="records")


print("✅ Handa na ang sanggunian at ang gate.")


In [ ]:
# 4. Tagutol ng Tagalog — sinusuri kung tunay na Tagalog ang naisulat.
#
# Walang diksyunaryo dito. Apat na senyales, at ang bawat isa ay sinukat laban sa
# totoong output ng 3B modelo:
#   1. Paulit-ulit na parirala (3-gram). Sa totoong basura, 27% ng mga 3-gram ay
#      paulit-ulit; sa malinis na screenplay, 1%.
#   2. Gawa-gawang salita. Mahigpit ito sa dalawang direksyon: hindi dapat
#      mabandera ang totoong "bahay-kubo", at dapat mahuli ang
#      "pagkakaiba-biba".
#   3. Kakulangan ng pantulong na salita (ang, ng, sa, si, ay).
#   4. ANYO. Ito ang pinakamalakas na senyales para sa endpoint ng screenplay, at
#      hindi ito tungkol sa bokabularyo: ang 3B modelo ay sumasagot ng sanaysay
#      na may "Simula/Gitna/Wakas" kahit hinihingi ng app ay EKSENA/VISUAL/
#      NARRATION. Ang app ay nagbabasa ng eksena, kaya ang sanaysay ay
#      nawawalang pelikula — kahit matino ang Tagalog nito.
import re

FUNCTION_WORDS = {
    "ang", "ng", "sa", "si", "ni", "kay", "na", "ay", "mga", "at",
    "siya", "niya", "kaniya", "ko", "mo", "namin", "natin", "sila", "nila",
    "ito", "iyan", "iyon", "may", "mayroon", "para", "dahil", "kung",
    "ako", "ikaw", "kami", "tayo", "hindi", "oo", "din", "rin", "lang",
}

# Tunay na unlapi sa Tagalog. Tama ang "nag-iisip" at "pag-ibig".
PREFIXES = {
    "nag", "mag", "pag", "ka", "ma", "pa", "na", "um", "in", "i", "ipa",
    "maki", "paki", "taga", "palang", "pinaka", "sang", "tig",
}

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
WORD_RE = re.compile(r"[A-Za-zÑñÁáÉéÍíÓóÚú'\-]+")
TERMINAL = ".!?\"')"
# Tinatanggap din ang "SCENE n": sumagot ang totoong modelo ng Ingles na
# pamagat ng eksena kahit Filipino ang katawan, at dating 0 ang bilang.
SCENE_RE = re.compile(r"\b(?:EKSENA|SCENE)\s*\d+", re.IGNORECASE)

MIN_SCENES = 3
MIN_TAGALOG_SCORE = int(os.environ.get("TAGALOG_MIN_SCORE", "55"))


def sentences(text):
    return [s.strip() for s in SENTENCE_SPLIT.split(text or "") if s.strip()]


def tokens(text):
    return [w.lower() for w in WORD_RE.findall(text or "")]


def _common_prefix(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def reduplication_errors(text):
    """Ibalik ang mga gawa-gawang salitang may gitling.

    Ang unang bersyon ng tuntuning ito ay nagbandera ng 13 sa 31 totoong salita:
    "bahay-kubo", "takip-silim", "ningas-kugon". Iyon ay higit na masama kaysa sa
    hindi pagbandera, dahil pinaparusahan nito ang tamang Tagalog. Ang tuntunin
    ngayon ay tumitingin sa HUGIS ng dalawang hati, hindi sa pagkakaiba ng
    bokabularyo:
      - magkaiba ang hati at kalahati lang ito ng naunang salita
        ("pagkakaiba-biba") -> gawa-gawa
      - magkapareho ang unang apat na letra ("masamang-masama") -> gawa-gawa
      - tunay na pag-uulit ng maikling salita ("araw-araw") -> tama
      - magkaibang salita ("bahay-kubo") -> tama
    """
    bad = []
    for word in WORD_RE.findall(text or ""):
        if "-" not in word:
            continue
        parts = word.lower().split("-")
        if any(p in PREFIXES for p in parts):
            continue
        if len(parts) != 2:
            continue
        left, right = parts
        if not left or not right:
            continue
        if left == right:
            # Ang pag-uulit ng buong mahabang salita ay hindi reduplikasyon.
            if len(left) > 6:
                bad.append(word)
            continue
        # Tunay na reduplikasyon na may linker: "unti-unting" = "unti" + "-ng",
        # "dahan-dahang" = "dahan" + "-ng". Nakita ito sa totoong output ng
        # libreng modelo at dating na-bandera bilang gawa-gawa. Ang haba ng
        # dagdag ay 1-2 letra lamang (ang linker), kaya hindi nito tinatakpan
        # ang tunay na basura tulad ng "pagkakaiba-biba".
        if right.startswith(left) and 0 < len(right) - len(left) <= 2:
            continue
        if len(right) >= 4 and left.endswith(right[-3:]):
            bad.append(word)
            continue
        if _common_prefix(left, right) >= 4 and abs(len(left) - len(right)) <= 3:
            bad.append(word)
    return bad


def repeated_phrases(text, n=3):
    """Bilang ng paulit-ulit na n-gram at ang kabuuang bilang ng n-gram.

    Buong pangungusap lang ang titignan ay kulang: sa totoong basura, iisa lang
    ang eksaktong naulit na pangungusap, pero 27% ng mga parirala ay naulit.
    """
    words = tokens(text)
    if len(words) < n:
        return 0, 0
    grams = [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    counts = {}
    for gram in grams:
        counts[gram] = counts.get(gram, 0) + 1
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated, len(grams)


def tagalog_report(text):
    """Sukatin ang kalidad ng wika. (puntos 0-100, listahan ng problema)."""
    issues = []
    lines = sentences(text)
    words = tokens(text)
    if not words:
        return 0, ["walang nilalaman"]

    repeated, grams = repeated_phrases(text)
    repeat_ratio = repeated / grams if grams else 0.0
    if repeat_ratio > 0.02:
        issues.append(f"paulit-ulit na parirala ({repeat_ratio:.0%} ng mga 3-gram)")

    invented = reduplication_errors(text)
    if invented:
        issues.append("gawa-gawang salita: " + ", ".join(sorted(set(invented))[:5]))

    ratio = sum(1 for w in words if w in FUNCTION_WORDS) / len(words)
    if len(words) >= 20 and ratio < 0.15:
        issues.append(f"kakaunti ang pantulong na salita ({ratio:.0%})")

    if lines and lines[-1][-1] not in TERMINAL:
        issues.append("hindi natapos ang huling pangungusap")

    score = 100
    score -= min(70, round(repeat_ratio * 250))
    score -= min(45, len(invented) * 25)
    if len(words) >= 20 and ratio < 0.15:
        score -= 20
    if lines and lines[-1][-1] not in TERMINAL:
        score -= 10
    return max(0, score), issues


MARKDOWN_RE = re.compile(r"^[\s>*_#`\-\u2022]+|[*_`]+$")


def normalize_line(line):
    """Alisin ang markdown bago suriin: "### **Eksena 1**" -> "Eksena 1"."""
    return MARKDOWN_RE.sub("", line.strip()).strip("*_` ").strip()


def screenplay_shape(text):
    """Bilangin ang hinihingi ng app: eksena, at ang VISUAL/NARRATION ng bawat isa.

    Ang mahalagang detalye: binibilang ang VISUAL at NARRATION na NAKAPALOOB sa
    ilalim ng isang EKSENA, hindi ang lahat ng "VISUAL:" sa teksto. Kung hindi,
    papasa ang isang sanaysay na may isang pekeng linya ng "VISUAL:" sa dulo.

    Ang linya ay nililinis muna ng markdown. Sumagot ang totoong modelo ng
    "### **Eksena 1 – ...**" at "**Pamagat:**", at dating 0 ang bilang ng eksena
    dahil sa mga asterisko — tinanggihan ang isang screenplay na tama naman ang
    anyo. Ang markdown ay presentasyon, hindi kahulugan.
    """
    shape = {"eksena": 0, "visual": 0, "narration": 0, "pamagat": 0}
    if not text:
        return shape
    lines = text.splitlines()
    in_scene = False
    for line in lines:
        upper = normalize_line(line).upper()
        if not upper:
            continue
        if SCENE_RE.match(upper):
            shape["eksena"] += 1
            in_scene = True
            continue
        if upper.startswith("PAMAGAT"):
            shape["pamagat"] += 1
            in_scene = False
            continue
        if in_scene and upper.startswith("VISUAL"):
            shape["visual"] += 1
        elif in_scene and upper.startswith("NARRATION"):
            shape["narration"] += 1
    return shape


def screenplay_report(text, min_scenes=MIN_SCENES):
    """(tanggap?, puntos, problema, hugis).

    Dalawang tanong ang magkaiba at parehong mahalaga:
      - tama ba ang wika?  -> tagalog_report
      - screenplay ba ito? -> screenplay_shape
    Ang sanaysay na may tamang Tagalog ay pumapasa sa wika pero bagsak sa anyo,
    at ang anyo ang siyang kailangan ng app. Dating iniuulat lang ang puntos at
    ipinapasa pa rin ang teksto; ngayon ay hinahatulan na.
    """
    score, issues = tagalog_report(text)
    shape = screenplay_shape(text)
    problems = []
    if shape["eksena"] < min_scenes:
        problems.append(f"{shape['eksena']} eksena lang (kailangan {min_scenes}+)")
    if shape["visual"] < shape["eksena"]:
        problems.append(f"kulang ang VISUAL ({shape['visual']}/{shape['eksena']})")
    if shape["narration"] < shape["eksena"]:
        problems.append(f"kulang ang NARRATION ({shape['narration']}/{shape['eksena']})")
    if not shape["pamagat"]:
        problems.append("walang PAMAGAT")
    # Putol na screenplay ay sirang pelikula, kaya hadlang ito at hindi lang
    # bawas sa puntos. Nangyari ito sa totoong libreng modelo: natapos ang sagot
    # sa gitna ng pangungusap ("...at sa pag-usbong nito,") at dating pumasa pa
    # rin, dahil 10 puntos lang ang bawas ng truncation.
    if "hindi natapos ang huling pangungusap" in issues:
        problems.append("putol ang screenplay sa dulo")
    ok = not problems and score >= MIN_TAGALOG_SCORE
    return ok, score, issues + problems, shape


print(f"✅ Handa na ang tagutol ng Tagalog (pinakamababa: {MIN_TAGALOG_SCORE}).")


In [ ]:
# 5. Kargahin ang modelo sa isang GPU.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("DIREK_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
MODEL_READY = False

print(f"⏳ Kinakarga ang {MODEL_ID}...")
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
# `dtype=` ang bagong pangalan; deprecated na ang `torch_dtype=`.
MODEL = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
MODEL.eval()
MODEL_READY = True
print(f"✅ Nakarga ang modelo sa {MODEL.device}")


In [ ]:
# 6. Pagbuo ng teksto + ang persona ni Direk.
#
# MAHALAGANG PASYA: repetition_penalty=1.0, ibig sabihin patay.
# Sa Tagalog, ang pinaka-madalas na token ay ang mga pantulong na salita — ang,
# ng, sa, na, ay. Kapag pinarusalan ng repetition_penalty ang mga paulit-ulit na
# token, pinipilit ang modelo na ipalit ang mga ito ng bihira at maling salita.
# Iyon ang dahilan ng "Nakakakaban talaga ang mga sariling puso".
import torch

DIREK_SYSTEM = (
    "Ikaw si Direk, isang direktor ng pelikula sa Pilipinas. Sumulat ka ng senaryo.\n"
    "Sumagot ka sa natural na Filipino, na parang nagkukuwento sa isang kaibigan.\n\n"
    "MAHALAGA: huwag kang mag-imbento ng salita. Kapag hindi mo alam ang salita, "
    "gamitin ang karaniwang salita na ginagamit sa bahay.\n"
    "Huwag mong ulitin ang isang pangungusap. Bawat pangungusap ay may bagong saysay.\n"
    "Huwag kang magsimula ng pangungusap sa 'pinaka-' o 'napaka-' maliban kung kailangan.\n\n"
    "HALIMBAWA NG TAMANG FILIPINO:\n"
    "Maaga pa lang ay gising na si Aling Nena. Naglalakad siya papunta sa palengke "
    "habang dala ang basket na puno ng kamatis. Tahimik ang kalsada, at malamig pa "
    "ang hangin. Doon niya nakita si Mang Tonyo, ang kapitbahay na nagtitinda ng isda.\n\n"
    "HALIMBAWA NG MALI (huwag gayahin):\n"
    "Ang mga tao ay hindi makakakuha ng sumusunod na kasaysayan. "
    "Masamang kasintahan ang mga pananampalataya na walang magbabago. "
    "Nakakakaban talaga ang mga sariling puso sa mga bayad na lumipas."
)


def gen(system, user, max_tokens=600, temperature=0.7):
    """Isang draft. Walang parusa sa mga paulit-ulit na salita."""
    if not MODEL_READY:
        raise RuntimeError("hindi pa nakarga ang modelo")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = TOKENIZER(prompt, return_tensors="pt").to(MODEL.device)
    with torch.no_grad():
        output = MODEL.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9,
            repetition_penalty=1.0,
            pad_token_id=TOKENIZER.eos_token_id,
        )
    text = TOKENIZER.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()


def gen_best(system, user, max_tokens=600, temperature=0.7, drafts=3):
    """Tatlong draft, pinakamalinis ang ibinabalik.

    Hindi pare-pareho ang kalidad ng isang 3B modelo. Mas matipid ang pumili ng
    pinakamalinis sa tatlo kaysa magtiwala sa isa.
    """
    best = None
    for _ in range(max(1, drafts)):
        text = gen(system, user, max_tokens=max_tokens, temperature=temperature)
        score, issues = tagalog_report(text)
        if best is None or score > best[0]:
            best = (score, issues, text)
        if score >= 90:
            break
    return {"text": best[2], "score": best[0], "issues": best[1]}


def build_script_prompt(brief, genre, duration, mood, titles):
    """Eksaktong hinihingi ng app: screenplay na may eksena, hindi sanaysay.

    Maikli ang prompt kaysa dati, at may isang buong halimbawa. Sa 3B modelo,
    ang isang konkretong halimbawa ay mas malakas kaysa sa limang tuntunin.
    """
    references = ""
    if titles:
        lines = [f"- {t['title']} ({t.get('release_year')}, {t.get('listed_in')})" for t in titles[:3]]
        references = "MGA KATULAD NA PELIKULA (huwag kopyahin, sanggunian lamang):\n" + "\n".join(lines) + "\n\n"
    return (
        f"Sumulat ng screenplay para sa isang pelikulang {genre or 'malaya ang uri'}. "
        f"Tagal: {duration or '1-2 minuto'}. Damdamin: {mood or 'natural'}.\n\n"
        f"{references}"
        "GAYAHIN ANG ANYO NA ITO NANG EKSAKTO:\n"
        "PAMAGAT: Ang Huling Bangka\n"
        "LOGLINE: Isang matandang mangingisda ang naghahanap sa kaniyang anak.\n"
        "EKSENA 1: Dalampasigan - Madaling araw\n"
        "  VISUAL: Nakatanaw si Isko sa dagat habang hawak ang lumang lambat.\n"
        "  NARRATION: Bawat umaga ay hinahanap niya ang bangkang nawala.\n"
        "EKSENA 2: Palengke - Umaga\n"
        "  VISUAL: Naglalakad si Isko sa masikip na palengke.\n"
        "  NARRATION: Tinatanong niya ang mga tindera kung may nakakita.\n\n"
        "Sumulat ng 6 na eksena. Bawat eksena ay may VISUAL at NARRATION. "
        "Huwag gumamit ng 'Simula', 'Gitna', o 'Wakas' bilang pamagat ng bahagi.\n\n"
        f"UTOS NG MANONOOD: {brief}"
    )


def build_scene_prompt(script):
    return (
        "Hatiin ang screenplay sa eksena. Sumagot ng JSON lamang, walang paliwanag.\n"
        'Anyong: {"scenes":[{"heading":"","narration":"","visual":"","mood":""}]}\n'
        "Panatilihin ang orihinal na Tagalog. Huwag magdagdag ng eksena na wala sa screenplay.\n\n"
        f"SCREENPLAY:\n{script}"
    )


print("✅ Handa na ang pagbuo ng teksto.")


In [ ]:
# 7. Ang FastAPI worker. Sinusunod ang kontrata ng app.
import json
import threading
import time
import uuid

import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(title="Direk Kaggle Worker")


def unavailable(reason):
    """Tapat na 503. Ginagawa itong PILOT_UNAVAILABLE ng app, hindi pekeng script."""
    return JSONResponse(
        {"error": reason, "code": "AI_UNAVAILABLE", "model": MODEL_ID},
        status_code=503,
    )


def rejected(reason):
    return JSONResponse({"error": reason, "code": "BAD_REQUEST", "model": MODEL_ID}, status_code=400)


class BriefReq(BaseModel):
    brief: str


class ScriptReq(BaseModel):
    script: str


class ChatReq(BaseModel):
    messages: list
    max_tokens: int = 600
    temperature: float = 0.7


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID, "model_ready": MODEL_READY, "persona": "Direk"}


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "direk"}]}


@app.post("/generate_script")
def generate_script(req: BriefReq):
    if not req.brief.strip():
        return rejected("Walang brief.")
    if not MODEL_READY:
        return unavailable("Hindi pa nakarga ang modelo.")
    genre = find_genre(req.brief)
    prompt = build_script_prompt(
        req.brief, genre, find_duration(req.brief), find_mood(req.brief), search_titles(genre)
    )
    try:
        result = gen_best(DIREK_SYSTEM, prompt, max_tokens=1200, temperature=0.7)
    except Exception as exc:
        return unavailable(f"Nabigo ang pagbuo ng screenplay: {exc}")
    ok, score, problems, shape = screenplay_report(result["text"])
    if not ok:
        # Tapat na 503. Mas mabuti pang walang pelikula kaysa pelikulang walang
        # eksena: ang app ay nagbabasa ng EKSENA/VISUAL/NARRATION, at ang
        # sanaysay ay nagiging zero-scene na pelikula nang tahimik.
        return unavailable("Hindi screenplay ang nabuo: " + "; ".join(problems))
    return {
        "script": result["text"],
        "model": MODEL_ID,
        "tagalog_score": score,
        "scenes_detected": shape["eksena"],
    }


@app.post("/generate_character_bible")
def generate_character_bible(req: ScriptReq):
    if not req.script.strip():
        return rejected("Walang screenplay.")
    if not MODEL_READY:
        return unavailable("Hindi pa nakarga ang modelo.")
    prompt = (
        "Ito ang screenplay:\n\n" + req.script + "\n\n"
        "Isulat ang character bible. Para sa bawat tauhan: PANGALAN, ITSURA, UGALI, BOSES.\n"
        "Isang talata bawat tauhan. Malinis at natural na Tagalog."
    )
    try:
        result = gen_best(DIREK_SYSTEM, prompt, max_tokens=1000, temperature=0.6)
    except Exception as exc:
        return unavailable(f"Nabigo ang pagbuo ng character bible: {exc}")
    if result["score"] < MIN_TAGALOG_SCORE:
        return unavailable("Mahina ang kalidad ng Tagalog: " + "; ".join(result["issues"]))
    return {"bible": result["text"], "model": MODEL_ID, "tagalog_score": result["score"]}


@app.post("/breakdown_scenes")
def breakdown_scenes(req: ScriptReq):
    if not req.script.strip():
        return rejected("Walang screenplay.")
    if not MODEL_READY:
        return unavailable("Hindi pa nakarga ang modelo.")
    try:
        raw = gen(
            "Ikaw ay tagapaghati ng screenplay. Sumagot ng JSON lamang.",
            build_scene_prompt(req.script),
            max_tokens=2000,
            temperature=0.2,
        )
    except Exception as exc:
        return unavailable(f"Nabigo ang paghahati sa eksena: {exc}")

    start, end = raw.find("{"), raw.rfind("}")
    if start < 0 or end <= start:
        return unavailable("Hindi JSON ang sagot ng modelo para sa eksena.")
    try:
        parsed = json.loads(raw[start:end + 1])
    except Exception:
        return unavailable("Sira ang JSON na sagot ng modelo para sa eksena.")

    scenes = parsed.get("scenes") if isinstance(parsed, dict) else None
    if not isinstance(scenes, list) or not scenes:
        return unavailable("Walang eksena sa sagot ng modelo.")

    # Ang app ay nagbabasa ng `visual`. Kung `visual_prompt` lang ang ilalabas,
    # mawawala ang biswal ng bawat eksena nang tahimik.
    clean = []
    for scene in scenes:
        if not isinstance(scene, dict):
            continue
        clean.append({
            "heading": str(scene.get("heading", "")).strip(),
            "narration": str(scene.get("narration", "")).strip(),
            "visual": str(scene.get("visual") or scene.get("visual_prompt", "")).strip(),
            "mood": str(scene.get("mood", "")).strip(),
        })
    if not clean:
        return unavailable("Walang wastong eksena sa sagot ng modelo.")
    return {"scenes": clean, "model": MODEL_ID}


@app.post("/v1/chat/completions")
def chat_completions(req: ChatReq):
    # Ang gate ay tumitingin sa LAHAT ng sinabi ng user, hindi lang sa huling
    # mensahe. Kung hindi, mawawala ang uri kapag sumagot siya ng "3 minuto".
    user_turns = [m.get("content", "") for m in req.messages if m.get("role") == "user"]
    whole = "\n".join(user_turns)
    genre = find_genre(whole)
    missing = missing_fields(whole)
    # UNA ang gate bago ang tsek ng modelo: purong string logic ito at wala itong
    # kailangang modelo. Kapag nauna ang tsek, hindi makapagtanong ang worker
    # habang naglo-load pa lang ang timbang, at magmumukhang sira ito.
    if missing:
        return {
            "id": "chatcmpl-" + uuid.uuid4().hex,
            "object": "chat.completion",
            "created": int(time.time()),
            "model": MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": QUESTIONS[missing[0]]},
                "finish_reason": "stop",
            }],
            "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
            "reference_genre": genre,
            "gate": missing[0],
        }

    if not MODEL_READY:
        return unavailable("Hindi pa nakarga ang modelo.")
    prompt = build_script_prompt(whole, genre, find_duration(whole), find_mood(whole), search_titles(genre))
    try:
        result = gen_best(DIREK_SYSTEM, prompt, max_tokens=req.max_tokens, temperature=req.temperature)
    except Exception as exc:
        return unavailable(f"Nabigo ang pagsagot: {exc}")
    return {
        "id": "chatcmpl-" + uuid.uuid4().hex,
        "object": "chat.completion",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": result["text"]},
            "finish_reason": "stop",
        }],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
        "reference_genre": genre,
        "gate": None,
        "tagalog_score": result["score"],
        # Sinasadya na hindi 503 dito: chat ito, at may mga tanong na sinasagot
        # ng maikling pangungusap na wala talagang eksena. Ang screenplay
        # endpoint ang naghahatol, dahil iyon ang binabasa ng app.
        "screenplay_ok": screenplay_report(result["text"])[0],
    }


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("✅ Tumatakbo ang FastAPI worker sa http://127.0.0.1:8000")


In [ ]:
# 8. Publikong tunnel. Mas gusto ang permanenteng named tunnel.
import os
import re
import subprocess
import time

CLOUDFLARED = "/kaggle/working/cloudflared"
if not os.path.exists(CLOUDFLARED):
    subprocess.run(
        ["wget", "-q", "-O", CLOUDFLARED,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
os.chmod(CLOUDFLARED, 0o755)

CF_TOKEN = os.environ.get("CF_TUNNEL_TOKEN")
CF_HOSTNAME = os.environ.get("CF_TUNNEL_HOSTNAME")
LOG = "/kaggle/working/cf.log"
PUBLIC_URL = None

if CF_TOKEN and CF_HOSTNAME:
    # Named tunnel: hindi nagbabago ang URL kahit i-restart ang session.
    cf_proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--no-autoupdate", "run", "--token", CF_TOKEN],
        stdout=open(LOG, "w"),
        stderr=subprocess.STDOUT,
    )
    PUBLIC_URL = CF_HOSTNAME.strip().rstrip("/")
    if not PUBLIC_URL.startswith("http"):
        PUBLIC_URL = "https://" + PUBLIC_URL
    print("✅ PERMANENTE ang URL (named tunnel).")
elif CF_TOKEN and not CF_HOSTNAME:
    raise RuntimeError(
        "May CF_TUNNEL_TOKEN pero walang CF_TUNNEL_HOSTNAME. "
        "Idagdag ang hostname ng named tunnel sa Kaggle Secrets."
    )
else:
    # Quick tunnel: libre, pero PANSAMANTALA — bago ang URL sa bawat session.
    cf_proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=open(LOG, "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        time.sleep(1)
        try:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(LOG).read())
        except FileNotFoundError:
            match = None
        if match:
            PUBLIC_URL = match.group(0)
            break
    print("⚠️ PANSAMANTALA ang URL (quick tunnel) — magbabago ito sa susunod na session.")

if PUBLIC_URL is None:
    raise RuntimeError("Walang tunnel URL pagkalipas ng 2 minuto. Tingnan ang /kaggle/working/cf.log")

with open("/kaggle/working/tunnel_url.txt", "w") as handle:
    handle.write(PUBLIC_URL)

print("✅ PUBLIC_URL:", PUBLIC_URL)


In [ ]:
# 9. Ipaalam sa app ang bagong URL, para hindi kailangang i-paste nang manwal.
import os

import requests

APP_URL = (os.environ.get("APP_URL") or "").strip().rstrip("/")
SYNC_TOKEN = os.environ.get("PILOT_SYNC_TOKEN")

if not APP_URL or not SYNC_TOKEN:
    print("ℹ️ Walang APP_URL o PILOT_SYNC_TOKEN; manwal na i-paste ang URL sa app.")
else:
    try:
        res = requests.post(
            APP_URL + "/api/pilot/update-kaggle-url",
            headers={"Authorization": f"Bearer {SYNC_TOKEN}"},
            json={"kaggle_url": PUBLIC_URL},
            timeout=30,
        )
        if res.ok:
            body = res.json()
            print("✅ Naka-save ang URL sa app:", body.get("kaggle_pilot_url"))
            if not body.get("durable", True):
                print("⚠️ Hindi ito mabubuhay pagkatapos ng restart ng app; i-save din sa settings.")
        else:
            print(f"⚠️ Tinanggol ng app ang URL ({res.status_code}): {res.text[:200]}")
    except Exception as exc:
        print(f"⚠️ Hindi naabot ang app: {exc}")


In [ ]:
# 10. Pagsusulit: totoong endpoints, at ang gate ng kulang na detalye.
import json
import socket

import requests
import urllib3.util.connection as urllib3_connection

# Pinipilit ang IPv4: may mga host na hindi umaabot sa IPv6 ng tunnel.
urllib3_connection.allowed_gai_family = lambda: socket.AF_INET
LOCAL_URL = "http://127.0.0.1:8000"

for attempt in range(6):
    try:
        health = requests.get(PUBLIC_URL + "/health", timeout=15)
        health.raise_for_status()
        print("✅ SAGOT NG TUNNEL:", health.status_code, health.json())
        break
    except Exception:
        if attempt == 5:
            raise
        print(f"⏳ Naghihintay sa DNS ({attempt + 1}/6)...")
        time.sleep(5)

# Ang gate: kulang ang detalye, kaya magtatanong muna — hindi mag-iimbento.
gate = requests.post(
    LOCAL_URL + "/v1/chat/completions",
    json={"messages": [{"role": "user", "content": "Gusto ko ng pelikula."}]},
    timeout=60,
).json()
print("\nGATE (dapat may tanong):", gate["choices"][0]["message"]["content"])
assert gate.get("gate"), "dapat nagtanong muna ang gate"

TESTS = [
    "Gumawa ka ng horror na kwento sa probinsya, 3 minuto, nakakakaba.",
    "Gumawa ka ng romance na kwento, 2 minuto, masaya.",
    "Gumawa ka ng comedy na kwento, 1 minuto, masaya, tungkol sa isang tindera sa palengke.",
]
for prompt in TESTS:
    data = requests.post(
        LOCAL_URL + "/v1/chat/completions",
        json={"messages": [{"role": "user", "content": prompt}], "max_tokens": 550, "temperature": 0.7},
        timeout=300,
    ).json()
    print(f"\nUSER: {prompt}")
    print(f"GATE: {data.get('gate')} | URI: {data.get('reference_genre')} | PUNTOS: {data.get('tagalog_score')}")
    print("DIREK:", data["choices"][0]["message"]["content"])

# Ang buong kontrata ng app, isa-isa.
script = requests.post(
    LOCAL_URL + "/generate_script",
    json={"brief": "Isang mangingisda sa probinsya, 2 minuto, malungkot."},
    timeout=300,
).json()
print("\n✅ /generate_script:", len(script.get("script", "")), "titik | puntos:", script.get("tagalog_score"))

scenes = requests.post(
    LOCAL_URL + "/breakdown_scenes", json={"script": script["script"]}, timeout=300
).json()
first = scenes["scenes"][0]
print("✅ /breakdown_scenes:", len(scenes["scenes"]), "eksena")
print("   unang eksena:", json.dumps(first, ensure_ascii=False)[:200])
assert first.get("visual"), "walang `visual` — mawawala ang biswal sa app"

fish_key = os.environ.get("FISHAUDIO_API_KEY")
if fish_key:
    fish = requests.post(
        "https://api.fish.audio/v1/tts",
        headers={"Authorization": f"Bearer {fish_key}", "Content-Type": "application/json", "model": "s2.1-pro-free"},
        json={"text": "Kumusta! Ako si Direk. Handa na ang ating kuwento."},
        timeout=30,
    )
    if fish.ok:
        with open("/kaggle/working/welcome.mp3", "wb") as handle:
            handle.write(fish.content)
        print("\n✅ Fish Audio: na-save ang /kaggle/working/welcome.mp3")
    else:
        print(f"\n⚠️ Fish Audio error {fish.status_code}: {fish.text[:150]}")
else:
    print("\nℹ️ Nilaktawan ang Fish Audio test.")


In [ ]:
# 11. Tsek ng mga sikreto: hindi kailanman ipinapakita ang halaga.
import os

REQUIRED = ["HF_TOKEN"]
OPTIONAL = [
    "FISHAUDIO_API_KEY",
    "CF_TUNNEL_TOKEN",
    "CF_TUNNEL_HOSTNAME",
    "GEMINI_API_KEY",
    "PILOT_SYNC_TOKEN",
    "APP_URL",
]

for name in REQUIRED:
    print(("✅" if os.environ.get(name) else "❌"), name, "(kailangan)")
for name in OPTIONAL:
    print(("✅" if os.environ.get(name) else "ℹ️"), name, "(opsyonal)")
print("\nHindi ipinapakita ang halaga ng kahit isang sikreto.")
